# Um agente que evolui: da busca ao agente com LLM
### Entregas no mundo Pokémon (Kanto e Johto)

**IESB · Inteligência Artificial (MDC050) · 2026/2 · Prof. Rodrigo Gonçalves**

| Fase | Responsável | O que o agente faz |
|---|---|---|
| 1. Busca | Ana Luiza | encontra a rota mais **curta** com A* (custo = distância) |
| 2. Aprendizado | Carlos Eduardo | aprende com dados o **tempo** de cada estrada; o mesmo A* passa a otimizar tempo previsto |
| 3. Agente com LLM | Lucas Pessoa | um LLM entende o pedido em português, **chama a busca como ferramenta** e explica a rota |

O agente final é uma peça só: o LLM (Fase 3) chama a busca (Fase 1), que usa os custos aprendidos (Fase 2).

**Como executar:** `Ambiente de execução → Alterar o tipo de ambiente → GPU (T4)` e depois `Ambiente de execução → Executar tudo`.
Nenhuma chave de API é usada: o LLM (Qwen2.5-1.5B-Instruct) roda dentro do próprio Colab. A GPU só é necessária na Fase 3; sem ela tudo funciona, mas cada pedido ao LLM leva de 30 a 60 s.

## 0. Preparação
Baixa o código do projeto e instala as dependências. Se o repositório for privado, cadastre um token do GitHub (somente leitura) nos *Secrets* do Colab (ícone de chave) com o nome `GITHUB_TOKEN`.

In [ ]:
import os, subprocess, sys

REPO_URL = "https://github.com/CarlosSouza-dev/IA-reserva.git"


def achar_raiz():
    for pasta in (os.getcwd(), os.path.dirname(os.getcwd())):
        if os.path.exists(os.path.join(pasta, "src", "agente_llm.py")):
            return pasta
    return None


RAIZ = achar_raiz()
if RAIZ is None:  # no Colab: clona o repositório
    url = REPO_URL
    try:
        from google.colab import userdata
        token = userdata.get("GITHUB_TOKEN")
        if token:
            url = REPO_URL.replace("https://", f"https://{token}@")
    except Exception:
        pass  # sem o Secret: funciona se o repositório for público
    subprocess.run(["git", "clone", "--depth", "1", url, "/content/IA-reserva"], check=True)
    RAIZ = "/content/IA-reserva"

os.chdir(RAIZ)
sys.path.insert(0, os.path.join(RAIZ, "src"))
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)

import matplotlib
matplotlib.use("Agg")  # os módulos salvam figuras; o notebook as exibe com display(Image)
import pandas as pd
from IPython.display import Image, Markdown, display

pd.set_option("display.max_colwidth", 120)
print("Projeto em", RAIZ)

## 1. O ambiente
O agente trabalha sobre um grafo de **36 locais** (cidades e pontos naturais) ligados por **45 estradas** de Kanto e Johto. As distâncias, em km, vêm do mapeamento das rotas do jogo para rodovias e balsas reais do Japão (documento de equivalência territorial do grupo).

**PEAS do agente:**
- **Desempenho:** entregar no menor tempo, por uma rota válida, respeitando as restrições do pedido.
- **Ambiente:** o mapa, com lombadas, semáforos, tipo de via, horário de pico e acidentes.
- **Atuadores:** escolher a próxima cidade da rota e responder ao usuário.
- **Sensores:** origem, destino(s), produto, horário e restrições do pedido.

In [ ]:
display(Image("docs/img/mapa_kanto_johto.jpeg", width=900))

from mapas import MAPA_GRANDE, MAPA_MEDIO, MAPA_PEQUENO, resumo_mapa, validar_mapa
for nome, mapa in [("Pequeno", MAPA_PEQUENO), ("Médio", MAPA_MEDIO), ("Grande", MAPA_GRANDE)]:
    nos, estradas = resumo_mapa(mapa)
    print(f"Mapa {nome}: {nos} nós, {estradas} estradas, problemas: {validar_mapa(mapa) or 'nenhum'}")

## 2. Fase 1: busca (rota mais curta)
**Formulação:** o *estado* é a cidade atual; o *estado inicial* é a origem; as *ações* são ir para uma cidade vizinha; a *transição* leva à cidade escolhida; o *teste de objetivo* é chegar ao destino; o *custo de caminho* é a soma das distâncias.

**Algoritmos:** busca de custo uniforme (não informada) e A*, com f(n) = g(n) + h(n).

**Heurística (distância direta, HDD):** h(n) = escala × distância em linha reta até o destino. A escala é *calibrada* no grafo: `escala = min(custo / distância_reta)` sobre todas as estradas. Assim, para toda estrada (a, b), custo(a, b) ≥ escala × reta(a, b), e pela desigualdade triangular h(a) ≤ c(a, b) + h(b). A heurística é **consistente**, portanto **admissível**.

*Correção em relação ao Marco 1:* a versão entregue dividia a distância por 10, o que **não** era admissível na estrada Saffron–Celadon (heurística 10 para custo real 9). O Mapa Médio tem 18 nós, não 21.

In [ ]:
from main import executar_fase1

benchmark = executar_fase1()
display(benchmark[["Mapa", "Algoritmo", "Custo Total", "Nós Expandidos", "Tempo (ms)"]])
display(Image("outputs/graficos/grafico_nos_expandidos.png", width=600))
display(Image("outputs/mapas/mapa_astar_grande_johto_kanto.png", width=800))

## 3. Fase 2: aprendizado (rota mais rápida de verdade)
O agente **não recebe** o tempo de cada estrada: ele aprende a partir de **2.700 travessias observadas** (45 estradas × 60 horários), com ruído de cerca de 8% e 2% de valores faltantes.

- **Pré-processamento** (tudo dentro de um `Pipeline`, ajustado só nos dados de treino): mediana para faltantes numéricos e moda para `tipo_via`, *one-hot* no tipo de via, hora em seno/cosseno (23h e 0h ficam vizinhas) e padronização.
- **Modelos:** baseline (média), Regressão Ridge, k-NN, Árvore de Decisão e Floresta Aleatória.
- **Validação cruzada em dois cenários:** *estradas conhecidas* (KFold), que é o uso real do agente, e *estradas novas* (GroupKFold por estrada), mais exigente e usado para escolher o modelo.

In [ ]:
from avaliacao_fase2 import executar_treino_e_avaliacao

preditor, comparacao = executar_treino_e_avaliacao()
display(comparacao[["Modelo", "MAE estradas conhecidas (min)", "R² estradas conhecidas",
                    "MAE estradas novas (min)", "R² estradas novas"]])
for grafico in ("fase2_comparacao_modelos", "fase2_previsto_vs_real", "fase2_erro_por_grupo"):
    display(Image(f"outputs/graficos/{grafico}.png", width=900))

**Onde o modelo erra** (resumo gerado automaticamente):

In [ ]:
display(Markdown(open("outputs/fase2_resumo.md", encoding="utf-8").read().split("## Onde o modelo erra mais")[1]))

### O modelo mudando a decisão
A **mesma** busca A* da Fase 1, agora com custo = tempo previsto. A tabela compara a rota mais curta (Fase 1), a rota escolhida pelo modelo e a rota ótima segundo o simulador (o gabarito, que o agente não conhece).

In [ ]:
from main import executar_fase2

fase2 = executar_fase2(preditor)
display(fase2[["Cenário", "Tempo real rota curta (min)", "Tempo real rota modelo (min)",
               "Economia vs curta (min)", "Mudou a rota?", "Modelo achou a ótima?"]])
display(Image("outputs/mapas/mapa_tempo_pewter_fuchsia_18h.png", width=800))

## 4. Fase 3: agente com LLM
O LLM **não calcula** a rota. Ele interpreta o pedido, escolhe uma **ferramenta** e preenche os argumentos (saída estruturada `<tool_call>`). O código executa a ferramenta (A* da Fase 1 com o custo da Fase 2), devolve o JSON ao LLM, e ele explica o resultado em português.

| Ferramenta | O que faz |
|---|---|
| `buscar_rota` | rota mais rápida com várias paradas, horário, cidades a evitar, acidentes e bloqueios |
| `estimar_tempo_trecho` | tempo previsto de uma estrada |
| `listar_vizinhas` / `listar_cidades` | consulta ao mapa |

**Proteções:** as ferramentas validam os nomes (corrigem apelidos e erros de digitação) e nunca levantam exceção. A resposta do LLM passa por `verificar_fidelidade`: se citar um número, uma cidade ou uma estrada que não veio da ferramenta, o texto é descartado e o agente usa uma explicação gerada direto do resultado.

Primeiro, as ferramentas sozinhas, sem LLM:

In [ ]:
import json
import ferramentas as F

F.definir_preditor(preditor)  # o modelo da Fase 2 é o custo das ferramentas
resultado = F.executar_ferramenta("buscar_rota", {
    "origem": "pallet", "paradas": ["Celadon", "Lavender"], "hora": "18h", "produto": "Poké Balls"})
print(json.dumps({k: resultado[k] for k in ("ordem_entregas", "rota", "tempo_previsto_min",
                                            "rota_mais_curta", "economia_vs_mais_curta_min")},
                 ensure_ascii=False, indent=2))
print(F.executar_ferramenta("buscar_rota", {"origem": "Pallet", "destino": "Hogwarts"}))

Agora o agente completo. O primeiro carregamento baixa o modelo, cerca de 3 GB.

In [ ]:
from agente_llm import PROMPT_PADRAO, criar_agente

agente = criar_agente()  # Qwen/Qwen2.5-1.5B-Instruct, prompt v3
print("Modelo:", agente.llm.nome, "| dispositivo:", agente.llm.dispositivo, "| prompt:", PROMPT_PADRAO)

In [ ]:
PEDIDOS_DEMO = [
    "Saio de Pallet às 18h e preciso entregar em Celadon e Lavender.",
    "Teve um acidente entre Saffron e Vermillion. Qual a rota de Pewter até Fuchsia às 18h?",
    "Me passa uma receita de bolo de cenoura?",
]
for pedido in PEDIDOS_DEMO:
    print("=" * 100, "\nPEDIDO:", pedido)
    r = agente.responder(pedido, mostrar=True)
    print("AGENTE:", r["resposta"])
    print(f"({r['segundos']} s, {r['tokens_entrada']}+{r['tokens_saida']} tokens"
          + (", resposta de reserva" if r["usou_reserva"] else "") + ")")

**Faça o seu pedido:** altere o texto e execute a célula.

In [ ]:
PEDIDO = "Leve 3 Poções de Goldenrod até Ecruteak agora, evitando Violet."
r = agente.responder(PEDIDO, mostrar=True)
print("\nAGENTE:", r["resposta"])

### A barreira contra alucinação
Casos reais da bateria: o Qwen escreveu números que a ferramenta não devolveu. A checagem detecta o problema e o agente troca o texto pela explicação gerada do resultado.

In [ ]:
from agente_llm import verificar_fidelidade

real = F.buscar_rota("Pewter", "Fuchsia", hora=18, acidentes=[["Saffron", "Vermillion"]])
alucinada = (f"Tempo previsto: {real['tempo_previsto_min']:.0f} min. "
             "Comparação: a rota mais curta tem 312 km; economia de 0.9 minutos.")
print("Texto do LLM :", alucinada)
print("Problemas    :", verificar_fidelidade(alucinada, [real]))
print("Ferramenta   : economia =", real["economia_vs_mais_curta_min"], "min")

## 5. Avaliação do agente: 20 pedidos de teste
Seguindo o Laboratório 10: os critérios foram escritos **antes** de rodar. São eles ferramenta certa, argumentos certos (comparados pelo significado), rota certa e resposta fiel. Os casos estão em `testes/pedidos_teste.json`.

Por padrão, esta seção carrega o resultado já salvo (CPU, prompt v3). Com `RODAR_BATERIA = True` ela roda de novo aqui: são 20 pedidos, poucos minutos na GPU.

In [ ]:
from avaliacao_agente import carregar_casos, executar_bateria, resumir, salvar, _tipos

RODAR_BATERIA = False

if RODAR_BATERIA:
    bateria = executar_bateria(agente, carregar_casos())
    salvar(bateria, f"{agente.llm.nome.split('/')[-1]} {agente.versao_prompt} colab")
else:
    bateria = _tipos(pd.read_csv("outputs/fase3/bateria_qwen2.5-1.5b-instruct_v3.csv"))

geral, por_categoria, matriz, precisao_revocacao = resumir(bateria)
display(pd.Series(geral, name="valor").to_frame())
display(por_categoria)
display(matriz)
display(precisao_revocacao)
display(bateria.loc[~bateria["acerto"], ["id", "categoria", "pedido", "diferencas_argumentos"]])

**Análise dos erros e tentativa de correção** (`outputs/fase3/analise_bateria_v3.md`):
- Os 3 erros do v3 foram argumentos que o modelo **deixou de extrair**: o retorno à origem, a estrada interditada (que virou o erro mais grave, porque a rota passou por ela) e o horário na consulta de um trecho.
- O prompt **v4** tentou corrigir esses erros com regras e exemplos novos e **derrubou o acerto de 85% para 50%**. A regra "pergunte a origem" fez o modelo pedir a origem até quando ela estava no pedido.
- Com um modelo pequeno, toda mudança de prompt precisa passar pela bateria inteira antes de ser adotada. O padrão continua sendo o v3.

## 6. IA responsável
- **Alucinação:** o LLM nunca calcula a rota, mas pode inventar números ao explicá-la. A checagem de fidelidade barrou 2 casos reais na bateria. Ela não detecta um número verdadeiro usado no lugar errado, nem afirmações factuais fora do domínio: com o prompt v4, o modelo respondeu "A Alemanha foi campeã da Copa de 2002", o que é falso.
- **Fora de escopo:** com o v3, os pedidos fora do domínio são recusados sem chamar ferramenta (receita, Copa, saudação), e destinos fora do mapa (São Paulo, Hogwarts) são recusados pela ferramenta, com explicação.
- **Custo e latência:** não há custo de API, porque o modelo roda localmente. Cada pedido usa cerca de 3.300 tokens, dos quais cerca de 90% são o prompt e a descrição das ferramentas. Na CPU, um pedido leva de 30 a 60 s; na GPU, poucos segundos.
- **Viés dos dados:** os tempos vêm de um simulador definido pelo grupo, então o modelo aprende as regras desse simulador, não o trânsito real. Ele erra mais em estradas longas e nunca vistas, porque árvores não extrapolam (detalhes na Fase 2).
- **LGPD:** o projeto não usa dados pessoais (cidades fictícias, pedidos de teste escritos pelo grupo). Com endereços e clientes reais, seria preciso definir a base legal, minimizar e anonimizar os pedidos registrados e não enviá-los a APIs externas sem consentimento. Aqui o LLM roda localmente, e nada sai do ambiente.

## 7. Bônus: conversa com o agente
Uma interface de texto simples, feita com `ipywidgets`.

In [ ]:
import ipywidgets as widgets

entrada = widgets.Text(placeholder="Ex.: Saio de Cerulean às 8h e entrego em Saffron e Celadon",
                       layout=widgets.Layout(width="80%"))
botao = widgets.Button(description="Enviar", button_style="primary")
saida = widgets.Output()


def ao_enviar(_):
    pedido = entrada.value.strip()
    if not pedido:
        return
    entrada.value = ""
    with saida:
        print("Você:", pedido)
        resposta = agente.responder(pedido)
        print("Agente:", resposta["resposta"], "\n")


botao.on_click(ao_enviar)
entrada.on_submit(ao_enviar)
display(widgets.HBox([entrada, botao]), saida)